In [1]:
from pathlib import Path
import os

_cwd = Path().resolve()
# Only go up a directory if we're in the notebooks dir
if os.path.basename(_cwd) == "notebooks":
    os.chdir(Path().resolve().parents[0])
 
print(os.getcwd())

C:\Users\OmkarSolankey\Downloads\Cineon\Project\test-binary-sdk


### Download the model

In [2]:
import mlflow

TRACKING_URI = "https://mlflow-server-479937931673.europe-west2.run.app"
mlflow.set_tracking_uri(TRACKING_URI)

# Use either a run artifact URI (runs:/...) or a model URI (models:/...).
# The MLflow UI page URL with '#/experiments/...' is not a downloadable artifact URI.
MODEL_URI = "models:/m-2bce1322d95d4673865cb192d4abbd52"

local_path = mlflow.artifacts.download_artifacts(
    artifact_uri=MODEL_URI,
    tracking_uri=TRACKING_URI,
    dst_path=r"model",
 )
print(f"Downloaded to: {local_path}")

c:\Users\OmkarSolankey\Downloads\Cineon\Project\test-binary-sdk\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

Downloaded to: C:\Users\OmkarSolankey\Downloads\Cineon\Project\test-binary-sdk\model\


### Model Inference

In [ ]:
from crossformer_binary_sdk.binary_sdk import BinarySDK

# Load the model from the local path where it was downloaded
sdk = BinarySDK()
sdk.load(r"model")

In [27]:
from cineon_format import CineonData

# Load the data from a CSV file and convert it to a CineonaData object, then to a DataFrame
data = CineonData.from_csv(r"C:\Users\OmkarSolankey\Desktop\lakefs_db\standardized_data\gd2\cineon-gd2-standardized-p034.csv")
data = data.to_dataframe()

# Select a subset of the data for testing and convert it back to a CineonData object
data = CineonData.from_dataframe(data.iloc[0:10000])
type(data)

cineon_format.cineon_data.CineonData

In [16]:
#  Preprocess the data and run it through the model
features, mask = sdk.preprocess([data])
logits = sdk.forward(features,mask)
print(f"features.shape: {features.shape}")
print(f"logits: {logits}")

features.shape: (4, 10, 51)
logits: [[ 0.28543752 -0.47654873  0.11571029]
 [ 0.02950043 -0.29504347  0.065005  ]
 [ 0.4196748  -0.7562206   0.46063533]
 [ 0.320447   -0.6122035   0.28545472]]


In [28]:
import numpy as np
import torch

# The entire model prediction pipeline can also be run in one step using the `run` method, which handles preprocessing and forward pass.
results = sdk.run([data])
print(f"results: {results}\n")
print(f"Logits: {results.logits}\n")
# get prediction probabilities
probabilities = torch.softmax(torch.tensor(results.logits), dim=1)
print(f"probabilities: {probabilities}\n")

predicted_classes = np.argmax(probabilities.numpy(), axis=1)
print(f"predicted_classes: {predicted_classes}")


results: <InferenceResult logits=(4, 3) features=(4, 10, 51) mask=(4, 10)>

Logits: [[ 0.28543752 -0.47654873  0.11571029]
 [ 0.02950043 -0.29504347  0.065005  ]
 [ 0.4196748  -0.7562206   0.46063533]
 [ 0.320447   -0.6122035   0.28545472]]

probabilities: tensor([[0.4328, 0.2020, 0.3652],
        [0.3625, 0.2620, 0.3756],
        [0.4255, 0.1313, 0.4433],
        [0.4239, 0.1668, 0.4093]])

predicted_classes: [0 2 2 0]
